<div style="
    width:100%;
    max-width:1080px;
    margin:24px 0 32px 0;
    background:linear-gradient(180deg, #1f2937 0%, #0f172a 100%);
    border:1px solid #334155;
    border-radius:22px;
    box-shadow:0 18px 38px rgba(15,23,42,0.24);
    overflow:hidden;
    color:#e6edf7;
">

<div style="
    height:8px;
    background:linear-gradient(90deg, #c2410c 0%, #f59e0b 100%);
"></div>

<div style="
    max-width:860px;
    padding:34px 38px 36px 38px;
    margin-left:0;
    margin-right:auto;
    font-family:'Aptos', 'Segoe UI Variable', 'Segoe UI', sans-serif;
    font-size:19px;
    line-height:1.78;
    color:#e6edf7;
">

<div style="margin-bottom:22px;">
<h1 style="margin:0 0 14px 0; font-size:34px; line-height:1.2; font-weight:700; color:#ffffff;">El Backpropagation con Autograd</h1>
<p style="margin:0; color:#cbd5e1; font-size:18px;">El aprendizaje de una red depende de poder propagar el error hacia atrás para ajustar sus parámetros. En este notebook se conecta esa idea con autograd, el mecanismo que PyTorch usa para calcular gradientes.</p>
</div>

<div style="display:grid; gap:14px;">
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Por qué importa</div>
    <div style="color:#f8fafc;">Backpropagation es el puente entre la predicción del modelo y la corrección de sus pesos.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Qué vas a aprender aquí</div>
    <div style="color:#f8fafc;">A entender cómo autograd sostiene la retropropagación y por qué los gradientes permiten actualizar la red.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Regla práctica que debes recordar</div>
    <div style="color:#f8fafc;">Si los gradientes no se calculan correctamente, la red no aprende aunque el resto del código parezca estar bien.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Contexto mínimo</div>
    <div style="color:#f8fafc;">Backpropagation aplica la regla de la cadena para repartir el error desde la salida hacia los parámetros internos.</div>
  </div>
</div>

</div>

</div>

<hr style="border:0; border-top:1px solid #64748b; opacity:0.22; margin:36px 0 42px 0;">

In [ ]:
# Bloque 1 — Creación de tensores de entrada y etiquetas
"""Este bloque crea tensores que representan las entradas y las etiquetas de un problema supervisado.
Este bloque importa "torch" para la creación y manipulación de tensores.
La variable "entradas" contiene muestras bidimensionales en forma de tensor.

Aplicamos el método ".unsqueeze()" a las etiquetas para que tengan una dimensión adicional, convirtiéndolas en tensores columna.
El argumento "1" indica que se debe agregar una dimensión adicional en la posición 1 del tensor.
La variable "etiquetas" almacena las etiquetas con la dimensión adicional en la posición 1.
Estos tensores permiten demostrar cómo autograd calcula gradientes a partir de la pérdida."""

import torch

entradas = torch.tensor([[1.0, 2.0],
                         [3.0, 4.0],
                         [5.0, 6.0]])

etiquetas = torch.tensor([0.0, 1.0, 0.0]).unsqueeze(1)

entradas , etiquetas  # Muestra los tensores de entrada y etiquetas con la dimensión adicional en etiquetas.


In [ ]:
# Bloque 2 — Definición de una red neuronal simple
"""Este bloque define una red neuronal artificial utilizando nn.Module.
Este bloque importa "torch.nn" para definir módulos y capas de redes neuronales.

Definimos una clase personalizada llamada "RedBackprop()" que representa un modelo de red neuronal simple.
La clase "RedBackprop()" hereda de "nn.Module", que es la clase base utilizada para construir redes neuronales en PyTorch.

Definimos la función "def __init__()", que inicializa la estructura interna del modelo.
El parámetro "self" de la función hace referencia a la instancia actual de la clase "RedBackprop()".

Utilizamos "super().__init__()" para invocar el constructor de la clase base "nn.Module".     
Esto asegura que la clase base se inicialice correctamente cada vez que se crea una instancia del modelo.

Aplicamos la clase "nn.Linear()" a la variable de instancia "self.capa1" para definir la primera capa lineal.
Los argumentos "2, 4" de "nn.Linear()" indican que la capa lineal recibe 2 entradas y produce 4 salidas.
Aplicamos la clase "nn.Linear()" a la variable de instancia "self.capa2" para definir la segunda capa lineal.
Los argumentos "4, 1" de "nn.Linear()" indican que la capa lineal recibe 4 entradas y produce 1 salida.

Definimos la función "def forward()" que especifica cómo fluye la entrada a través de las capas del modelo.
El método ".forward()" especifica cómo se propaga la información a través de la red.
Los parámetros "self" y "x" representan la instancia del modelo y los datos de entrada, respectivamente.

Asignamos el resultado de llamar a "self.capa1()" a la variable "x", aplicando la primera capa lineal a la entrada.
Aplicamos la función de activación "torch.relu()" a la variable "x" para introducir no linealidad en la red.
Asignamos el resultado de llamar a "self.capa2()" a la variable "x", aplicando la segunda capa lineal a la salida de la activación.
Utilizamos la instrucción "return" para devolver la salida del modelo aplicando las capas lineales a la entrada.

En todos los casos se pasa "x" como parámetro a las funciones correspondientes para que la información fluya a través de la red.
Además, la variable "x" se actualiza con la salida de cada operación, asegurando que la información fluya correctamente a través de la red.

La variable "red" almacena una instancia de la clase "RedBackprop()", representando el modelo de red neuronal definido.
La estructura permite demostrar cómo se calculan gradientes sobre los parámetros de la red."""

import torch.nn as nn

class RedBackprop(nn.Module):
    def __init__(self):
        super().__init__()
        self.capa1 = nn.Linear(2, 4)
        self.capa2 = nn.Linear(4, 1)

    def forward(self, x):
        x = self.capa1(x)
        x = torch.relu(x)
        x = self.capa2(x)
        return x

red = RedBackprop()    

red # Muestra la estructura de la red neuronal definida.


In [ ]:
# Bloque 3 — Forward pass
"""Este bloque ejecuta el forward pass de la red neuronal.
Asignamos el resultado de llamar a la instancia de la clase a la variable "predicciones" mediante la expresión "red(entradas)".
El argumento "entradas" representa los datos de entrada que se pasan a la red para generar las predicciones.
La variable "predicciones" almacena los logits generados por la red aún no entrenada, sin aplicar una sigmoide.
El forward pass construye el grafo computacional que autograd utilizará para calcular gradientes."""

predicciones = red(entradas)

predicciones # Muestra las salidas generadas por la red.


In [ ]:
# Bloque 4 — Cálculo de la función de pérdida
"""Este bloque calcula la función de pérdida entre las predicciones y las etiquetas.
Aplicamos la clase "nn.BCEWithLogitsLoss()" a la variable "criterio" para crear un objeto calculador de función de pérdida.
La variable "criterio" almacena el objeto calculador de la función de pérdida.

Asignamos la llamada a la función de pérdida a la variable "perdida" mediante la expresión "criterio(predicciones, etiquetas)".
El argumento "predicciones" representa los logits generados por la red; la función de pérdida aplica internamente la sigmoide.
El argumento "etiquetas" representa las etiquetas transformadas.

La variable "perdida" almacena el valor de la función de pérdida calculada entre las predicciones y las etiquetas.
El cálculo de la pérdida determina el error que autograd utilizará para generar gradientes."""

criterio = nn.BCEWithLogitsLoss()
perdida = criterio(predicciones, etiquetas)

criterio, perdida # Muestra el objeto calculador de la función de pérdida y el valor de la pérdida calculada.


In [ ]:
# Bloque 5 — Backward pass con autograd
"""Este bloque ejecuta el backward pass utilizando autograd.
Aplicamos el método "backward()" a la variable "perdida" para calcular los gradientes de la pérdida respecto a los parámetros de la red.
El método "backward()" calcula los gradientes de la pérdida respecto a los parámetros de la red.
Los gradientes quedan almacenados en los atributos grad de cada parámetro."""

perdida.backward()

# Autograd ha calculado los gradientes y los ha almacenado en los atributos grad de cada parámetro.

In [ ]:
# Bloque 6 — Inspección de gradientes
"""Este bloque inspecciona los gradientes generados por autograd.
Asignamos la expresión "red.capa1.weight.grad" a la variable "gradientes_capa1" para obtener los gradientes de los pesos de la primera capa.
La variable "gradientes_capa1" almacena los gradientes de los pesos de la primera capa.

Asignamos la expresión "red.capa2.weight.grad" a la variable "gradientes_capa2" para obtener los gradientes de los pesos de la segunda capa.
La variable "gradientes_capa2" almacena los gradientes de los pesos de la segunda capa.
La inspección permite observar los gradientes calculados por autograd para los pesos."""

gradientes_capa1 = red.capa1.weight.grad
gradientes_capa2 = red.capa2.weight.grad

gradientes_capa1, gradientes_capa2 # Muestra los gradientes de los pesos de la primera y segunda capa, respectivamente.


In [ ]:
# Bloque 7 — Estructura final de resultados
"""Este bloque organiza los elementos principales del proceso de backpropagation con autograd.
La variable "resultado" almacena en forma de diccionario las predicciones, la pérdida y los gradientes calculados.
La estructura permite visualizar el funcionamiento completo del mecanismo de autograd."""

resultado = {
    "predicciones": predicciones.tolist(),
    "perdida": float(perdida.item()), # El método item() obtiene el valor escalar de la pérdida.
    "gradientes_capa1": gradientes_capa1.tolist(),
    "gradientes_capa2": gradientes_capa2.tolist()
}

resultado # Muestra el diccionario con las predicciones, la pérdida y los gradientes de las capas.


<div style="
    width:100%;
    max-width:1080px;
    margin:40px 0 24px 0;
    background:linear-gradient(180deg, #1f2937 0%, #0f172a 100%);
    border:1px solid #334155;
    border-radius:22px;
    box-shadow:0 18px 38px rgba(15,23,42,0.24);
    overflow:hidden;
    color:#e6edf7;
">

<div style="
    height:8px;
    background:linear-gradient(90deg, #0f766e 0%, #2dd4bf 100%);
"></div>

<div style="
    max-width:860px;
    padding:34px 38px 36px 38px;
    margin-left:0;
    margin-right:auto;
    font-family:'Aptos', 'Segoe UI Variable', 'Segoe UI', sans-serif;
    font-size:19px;
    line-height:1.78;
    color:#e6edf7;
">

<div style="margin-bottom:22px;">
<h2 style="margin:0 0 14px 0; font-size:30px; line-height:1.2; font-weight:700; color:#ffffff;">Conclusiones</h2>
<p style="margin:0; color:#cbd5e1; font-size:18px;">En este notebook has visto cómo el error del modelo se traduce en gradientes útiles para corregir parámetros.</p>
</div>

<div style="display:grid; gap:14px;">
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Qué conviene recordar</div>
    <div style="color:#f8fafc;">La retropropagación no es un paso separado del aprendizaje, sino su mecanismo central.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Error frecuente a evitar</div>
    <div style="color:#f8fafc;">Interpretar los gradientes como números abstractos sin relacionarlos con la dirección de ajuste de los pesos.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Siguiente paso natural</div>
    <div style="color:#f8fafc;">Detenerse en las Funciones de Activación para dar continuidad directa al hilo del temario.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Conexión con el siguiente concepto</div>
    <div style="color:#f8fafc;">El siguiente notebook se centra en las Funciones de Activación y amplía de forma natural lo que acabas de trabajar.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Siguiente notebook del temario</div>
    <div style="color:#f8fafc;">04_Redes_neuronales_artificiales/04_Las_funciones_de_activacion.ipynb</div>
  </div>
</div>

</div>

</div>

<hr style="border:0; border-top:1px solid #64748b; opacity:0.22; margin:42px 0 24px 0;">